In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO
import time

from video_pr.video_preprocess import sliding_window

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
model = YOLO("best_weights_path.pt")
model.to("cuda")
print(model.device)

cuda:0


In [ ]:
video_path = "/Stock_video/your_video.mp4"
window_size = 30
overlap = 15

In [ ]:
frame_gen = sliding_window(video_path, window_size, overlap)
frame = next(frame_gen)
frame = frame["data"][0]

In [ ]:
def gen_maker():
  '''
  Функция для рестарта генератора с начала видео.
  '''
  return sliding_window(video_path, window_size, overlap)

In [ ]:
def corners_sort(corners):

    '''
    Сначала была идея пойти от центроид и построить сортировку углов на if, elif, else, но такой подход будет слабым при экстремальной перспективе.
    Если камера будет стоять под углом на площадку, то все может поломаться, так как Y верхнего правого угла может стать меньше Y правого нижнего.
    Напрмиер при проекции трапецевидного паралеллипипеда, а не простой трапеции.

    Поэтому решил считать в радианах через arctan2. После раскладывать углы "по кругу" и собирать начиная с Top Left (TL)
    '''

    center = np.mean(corners, axis=0)

    angles = np.arctan2(corners[:,1] - center[1],
                        corners[:,0] - center[0])

    sorted_idx = np.argsort(angles)

    corners = corners[sorted_idx]

    sums = corners[:,0] + corners[:,1]
    start_idx = np.argmin(sums)

    corners = np.roll(corners, -start_idx, axis=0)

    return corners

In [ ]:
def court_zone(model):
    '''Функция детекции волейбольной площадки

    Принцип работы:
    Обрабатывает кадры видео пока кол-в валидных кадров (valid_frames) < необходимого кол-ва кадров (total_frames = 450)
    Забирает из генератора словарь, в нем по ключу ["data"] забирает фреймы. Ищет и определяет на нем углы площадки.

    Если кол-во углов не равно 4 (if len(corners) != 4) пропускает кадр.

    В ином слуачае отправляет углы в corners_sort для поддержания порядка углов.

    Углы после сортировки сохраняет в corners_court (вид [[],[],[],...[]]
    и считается медианна между всеми углами каждой зоны (top left и тд)

    возвращает median_corners нампай массив с 8 точками координат углов (x, y каждого угла)
    '''

    processed_frames = 0
    valid_frames = 0

    total_frames = 450

    corners_court = []

    frame_gen = gen_maker() #Пересоздаем генератор при каждом запуске, чтобы запрашивать каждый раз первые 450 кадров

    start = time.time()

    break_outer = False

    while valid_frames < total_frames:
      frames = next(frame_gen)

      for frame in frames["data"]:
        processed_frames += 1
        print(f"Обрабатывается кадр №{processed_frames}")

        court_detect = model(frame)[0]

        if court_detect.masks is None:
          continue

        polygon_points = court_detect.masks.xy[0]
        contour = polygon_points.astype(np.int32)

        perimeter = cv2.arcLength(contour, True)

        e = 0.02 * perimeter
        approx_cornes = cv2.approxPolyDP(contour, e, True)

        corners = approx_cornes.reshape(-1, 2)

        if len(corners) != 4:
          continue

        corners = corners_sort(corners)

        corners_court.append(corners)
        valid_frames += 1

        if valid_frames >= total_frames:
          break_outer = True
          break

      if break_outer:
        break

    end = time.time()

    print(f"Обработано кадров: {processed_frames}")
    print(f"Валидных кадров: {valid_frames}")
    print(f"Время: {round(end - start, 1)} сек")

    corners_court = np.array(corners_court)

    median_corners = np.median(corners_court, axis=0)

    return median_corners

In [ ]:
court_cords = court_zone(model)

Обрабатывается кадр №1

0: 384x640 1 court, 54.7ms
Speed: 12.3ms preprocess, 54.7ms inference, 44.0ms postprocess per image at shape (1, 3, 384, 640)
Обрабатывается кадр №2

0: 384x640 1 court, 9.2ms
Speed: 2.7ms preprocess, 9.2ms inference, 2.1ms postprocess per image at shape (1, 3, 384, 640)
Обрабатывается кадр №3

0: 384x640 1 court, 7.8ms
Speed: 2.0ms preprocess, 7.8ms inference, 1.8ms postprocess per image at shape (1, 3, 384, 640)
Обрабатывается кадр №4

0: 384x640 1 court, 8.1ms
Speed: 2.8ms preprocess, 8.1ms inference, 1.9ms postprocess per image at shape (1, 3, 384, 640)
Обрабатывается кадр №5

0: 384x640 1 court, 7.5ms
Speed: 3.3ms preprocess, 7.5ms inference, 1.8ms postprocess per image at shape (1, 3, 384, 640)
Обрабатывается кадр №6

0: 384x640 1 court, 8.4ms
Speed: 2.0ms preprocess, 8.4ms inference, 1.9ms postprocess per image at shape (1, 3, 384, 640)
Обрабатывается кадр №7

0: 384x640 1 court, 7.4ms
Speed: 2.4ms preprocess, 7.4ms inference, 1.7ms postprocess per image 

In [ ]:
'''Перевод площадки из системы координат площадки в физическую систему координат

Размер "мира" 1900 : 2800 (worldld_size = (1900, 2800) где 1 пиксель равен 1 сантиметру реального мира

перевод выполняется за счет гомографии (cv2.findHomography)'''

physical_cords = np.array([[500,500],
                           [1400, 500],
                           [1400, 2300],
                           [500, 2300]
                        ], dtype=np.float32)


world_size = (1900, 2800)

H, mask = cv2.findHomography(court_cords, physical_cords, method = cv2.RANSAC )

# physical_court = cv2.warpPerspective(frame, H, world_size)

In [ ]:
'''Основные объекты площадки – ауты, зоны, линии атаки, сетка и тд'''

court_objects = {
    "court": np.array([
        [500, 500],
        [1400, 500],
        [1400, 2300],
        [500, 2300]
    ], dtype=np.float32),

    "out_zone": np.array([
        [0, 0],
        [1900, 0],
        [1900, 2800],
        [0, 2800]
    ], dtype=np.float32),

    "net": np.array([
        [500, 1400],
        [1400, 1400]
    ], dtype=np.float32),

    "attack_line_top": np.array([
        [500, 1100],
        [1400, 1100]
    ], dtype=np.float32),

    "attack_line_bottom": np.array([
        [500, 1700],
        [1400, 1700]
    ], dtype=np.float32),

    "zones_top": {

                  "zone_1": np.array([
                      [500, 500],
                      [800, 500],
                      [800, 950],
                      [500, 950]
                  ], dtype=np.float32),

                  "zone_2": np.array([
                      [500, 950],
                      [800, 950],
                      [800, 1400],
                      [500, 1400]
                  ], dtype=np.float32),


                  "zone_3": np.array([
                      [800, 950],
                      [1100, 950],
                      [1100, 1400],
                      [800, 1400]
                  ], dtype=np.float32),


                  "zone_4": np.array([
                      [1100, 950],
                      [1400, 950],
                      [1400, 1400],
                      [1100, 1400]
                  ], dtype=np.float32),


                  "zone_5": np.array([
                      [1100, 500],
                      [1400, 500],
                      [1400, 950],
                      [1100, 950]
                  ], dtype=np.float32),

                  "zone_6": np.array([
                      [800, 500],
                      [1100, 500],
                      [1100, 950],
                      [800, 950]
                  ], dtype=np.float32)
                  },


    "zones_bottom": {

                    "zone_1": np.array([
                        [1100, 1850],
                        [1400, 1850],
                        [1400, 2300],
                        [1100, 2300]
                    ], dtype=np.float32),


                    "zone_2": np.array([
                        [1100, 1400],
                        [1400, 1400],
                        [1400, 1850],
                        [1100, 1850]
                    ], dtype=np.float32),


                    "zone_3": np.array([
                        [800, 1400],
                        [1100, 1400],
                        [1100, 1850],
                        [800, 1850]
                    ], dtype=np.float32),

                    "zone_4": np.array([
                        [500, 1400],
                        [800, 1400],
                        [800, 1850],
                        [500, 1850]
                    ], dtype=np.float32),

                    "zone_5": np.array([
                        [500, 1850],
                        [800, 1850],
                        [800, 2300],
                        [500, 2300]
                    ], dtype=np.float32),

                    "zone_6": np.array([
                        [800, 1850],
                        [1100, 1850],
                        [1100, 2300],
                        [800, 2300]
                    ], dtype=np.float32)
                }
}
